# Day 14: Data Pipelines and Spark Intro
**Batch pipelines, Spark overview, DataFrames, transformations and joins**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
Colab includes Java, so PySpark works after a quick install. In VS Code, install **Java 17** (for example Eclipse Temurin) and then `pip install pyspark`.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q pyspark

# ===== Option B: VS Code / local (run in your terminal once) =====
# 1. Install Java 17 and check:  java -version
# 2. pip install pyspark
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("day14")
         .config("spark.sql.shuffle.partitions", "4")   # small data, fewer partitions
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 2. Raw data (bronze input)

In [ ]:
%%writefile orders.csv
order_id,customer_id,amount,order_date,city
1001,1,250.50,2026-11-12, Bengaluru
1002,2,1200,2026-11-12,Tokyo
1002,2,1200,2026-11-12,Tokyo
1003,3,,2026-11-12,Osaka
1004,4,-20,2026-11-13,Pune
1005,5,560,2026-11-13,Chennai
1006,6,90.25,2026-11-13,Kyoto
1007,1,3100,2026-11-13,Bengaluru
1008,2,3300,2026-11-12,Tokyo
1009,99,780,2026-11-14,Pune
1010,98,150,2026-11-14,Osaka
1011,3,410,2026-11-14,Osaka

In [ ]:
%%writefile customers.csv
customer_id,name,segment
1,Asha,retail
2,Kenji,enterprise
3,Mei,retail
4,Ravi,retail
5,Priya,enterprise
6,Hana,retail

In [ ]:
orders_raw = spark.read.csv("orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
orders_raw.printSchema()
orders_raw.show(5)
print("Raw orders:", orders_raw.count(), "| Customers:", customers.count())

## 3. Lazy evaluation in action

In [ ]:
import time
t = time.perf_counter()
plan = (orders_raw.filter(F.col("amount") > 0)
        .withColumn("amount_tax", F.col("amount") * 1.18)
        .select("order_id", "city", "amount_tax"))
print(f"Transformations: {time.perf_counter() - t:.3f} s  (nothing ran yet)")
t = time.perf_counter()
print("Rows:", plan.count())
print(f"Action count(): {time.perf_counter() - t:.3f} s  (the whole plan ran)")

## 4. Silver: clean and enrich

In [ ]:
silver = (orders_raw
    .dropDuplicates(["order_id"])
    .filter(F.col("amount").isNotNull() & (F.col("amount") > 0))
    .withColumn("city", F.trim("city"))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("size", F.when(F.col("amount") >= 1000, "large")
                         .when(F.col("amount") >= 300, "medium")
                         .otherwise("small")))
silver.orderBy("order_id").show()
print("Silver rows:", silver.count())

### TODO 1: Add a column `weekday` using `F.date_format("order_date", "E")` and count orders per weekday.

In [ ]:
# TODO: write your code here

## 5. Joins: check counts every time

In [ ]:
inner = silver.join(customers, "customer_id", "inner")
left = silver.join(customers, "customer_id", "left")
unknown = silver.join(customers, "customer_id", "left_anti")

print("silver:", silver.count(), "| inner:", inner.count(), "| left:", left.count())
print("Orders with unknown customer:", [r.order_id for r in unknown.orderBy("order_id").collect()])
left.select("order_id", "name", "segment", "amount").orderBy("order_id").show()

## 6. Gold: aggregate

In [ ]:
gold = (left
    .groupBy("order_date", "city")
    .agg(F.count("*").alias("orders"),
         F.round(F.sum("amount"), 2).alias("revenue"),
         F.round(F.avg("amount"), 2).alias("avg_order"))
    .orderBy("order_date", F.desc("revenue")))
gold.show()

by_segment = (left.fillna({"segment": "unknown"})
              .groupBy("segment").agg(F.round(F.sum("amount"), 2).alias("revenue"))
              .orderBy(F.desc("revenue")))
by_segment.show()

## 7. Write the medallion layers as Parquet

In [ ]:
orders_raw.write.mode("overwrite").parquet("lake/bronze/orders")
silver.write.mode("overwrite").parquet("lake/silver/orders")
gold.write.mode("overwrite").partitionBy("order_date").parquet("lake/gold/daily_city_revenue")

import os
print(sorted(os.listdir("lake/gold/daily_city_revenue")))
back = spark.read.parquet("lake/gold/daily_city_revenue")
print("Gold rows:", back.count())
back.filter("order_date = '2026-11-13'").show()

## 8. Window function: top city per day

In [ ]:
from pyspark.sql.window import Window
w = Window.partitionBy("order_date").orderBy(F.desc("revenue"))
(gold.withColumn("rank", F.rank().over(w))
     .filter("rank = 1")
     .select("order_date", "city", "revenue")
     .orderBy("order_date")
     .show())

### TODO 2: Re-run section 7 and confirm the gold row count is unchanged. Then try `mode("append")` and see what happens to the count. Which mode makes the job idempotent?

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)